# पाठ 05 - एजेंटिक RAG


## Setup

यह नोटबुक Microsoft Agent Framework का उपयोग करके Agentic RAG (Retrieval-Augmented Generation) पैटर्न को प्रदर्शित करती है।

**पूर्व आवश्यकताएं:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — आपकी Azure AI Search सेवा का endpoint
- `AZURE_SEARCH_API_KEY` — आपकी Azure AI Search API कुंजी
- पर्यावरण मानदंडों के माध्यम से Azure OpenAI तैनाती कॉन्फ़िगर की गई
- Azure CLI प्रमाणित (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## एजेंटिक RAG क्या है?

पारंपरिक RAG एक नियत पाइपलाइन का पालन करता है: दस्तावेज़ प्राप्त करें, फिर उत्तर उत्पन्न करें। **एजेंटिक RAG** एक कदम आगे जाता है, एजेंट को स्वायत्तता देता है यह तय करने की कि **कब** और **कैसे** सूचना प्राप्त की जाए।

एजेंटिक RAG के साथ, एजेंट कर सकता है:
- **निर्णय लें** कि प्रश्न का उत्तर देने से पहले पुनःप्राप्ति आवश्यक है या नहीं
- **चुनें** कि कौन सा डेटा स्रोत या उपकरण क्वेरी करना है
- **आकलन करें** प्राप्त परिणामों का और यदि पहला प्रयास अपर्याप्त हो तो फ़ॉलो-अप पुनःप्राप्ति करें
- **संयुक्त करें** कई पुनःप्राप्ति चरणों से जानकारी को एक सुसंगत उत्तर में

यह एजेंट को एक स्थिर पुनःप्राप्ति-फिर-उत्पादन पाइपलाइन की तुलना में अधिक लचीला और सटीक बनाता है।


## एक खोज उपकरण बनाना

Agentic RAG में, बाहरी डेटा स्रोतों को **टूल्स** के रूप में लपेटा जाता है जिन्हें एजेंट आवश्यकतानुसार कॉल कर सकता है। इससे एजेंट को पुनःप्राप्ति को सिर्फ एक और क्रिया के रूप में लेने की अनुमति मिलती है, न कि एक अनिवार्य चरण के रूप में।

नीचे हम एक यात्रा ज्ञान आधार परिभाषित करते हैं और इसे एक टूल के रूप में एक्सपोज़ करते हैं जिसे एजेंट गंतव्य जानकारी देखने के लिए कॉल कर सकता है।


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## RAG एजेंट बनाना

अब हम एक एजेंट बनाते हैं जिसे निर्देश दिया गया है कि **सदैव उत्तर देने से पहले जानकारी प्राप्त करें**। एजेंट अपने उत्तरों को प्रशिक्षण डेटा पर भरोसा करने के बजाय ज्ञान आधार में आधारित करने के लिए `search_travel_knowledge` टूल का उपयोग करता है।


In [ ]:
agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## पुनरावृत्त Retrieval — मेकर-चेककर पैटर्न

Agentic RAG का एक प्रमुख लाभ **पुनरावृत्त रिट्रीवल** है। एजेंट अपने शुरुआती निष्कर्षों को सत्यापित, परिष्कृत या विस्तारित करने के लिए कई राउंड की खोज कर सकता है — जो "मेकर-चेककर" वर्कफ़्लो के समान है:

1. **मेकर कदम**: एजेंट प्रारंभिक जानकारी प्राप्त करता है और एक उत्तर का मसौदा तैयार करता है।
2. **चेककर कदम**: एजेंट विवरणों की जांच करने या खामियों को भरने के लिए अतिरिक्त रिट्रीवल करता है।

नीचे, एजेंट से एक प्रश्न पूछा गया है जिसमें कई गंतव्यों की तुलना करनी है, जिससे इसे कई बार खोज करने के लिए प्रेरित किया जाता है।


In [ ]:
checker_agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## सारांश

इस पाठ में आपने सीखा कि Microsoft Agent Framework का उपयोग करके **Agentic RAG** सिस्टम कैसे बनाया जाता है:

- **Agentic RAG** एजेंटों को स्वायत्त रूप से यह तय करने देता है कि कब जानकारी प्राप्त करनी है, जिससे प्राप्ति स्थिर की बजाय गतिशील हो जाती है।
- **टूल्स को डेटा स्रोत के रूप में**: बाहरी ज्ञान आधार (जैसे Azure AI Search) टूल्स के रूप में लिपटे होते हैं जिन्हें एजेंट कॉल कर सकता है।
- **आवृत्तिमूलक प्राप्ति**: मेकर-चेककर पैटर्न एजेंट को कई बार प्राप्ति करने की अनुमति देता है — खोज करना, सत्यापित करना, और सुधारना — अंतिम उत्तर प्रदान करने से पहले।

उत्पादन में, आप बड़े पैमाने पर यात्रा दस्तावेज़ प्राप्ति को संभालने के लिए मेमोरी में संग्रहित `TRAVEL_KNOWLEDGE_BASE` को असली Azure AI Search इंडेक्स से बदलेंगे।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:  
यह दस्तावेज़ AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) का उपयोग करके अनूदित किया गया है। जबकि हम सटीकता के लिए प्रयासरत हैं, कृपया ध्यान दें कि स्वचालित अनुवादों में त्रुटियाँ या गलतियां हो सकती हैं। मूल दस्तावेज़ अपनी मूल भाषा में अधिकारिक स्रोत माना जाना चाहिए। महत्वपूर्ण जानकारी के लिए, पेशेवर मानव अनुवाद की सलाह दी जाती है। इस अनुवाद के उपयोग से उत्पन्न किसी भी गलतफहमी या गलत व्याख्या के लिए हम जिम्मेदार नहीं हैं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
